# TxGNN 演示：基于知识图谱的零样本药物重定向

> **配套教材**：第六章 · 知识图谱与药物重定向  
> **原始论文**：[Zero-shot Prediction of Therapeutic Use with Geometric Deep Learning](https://www.medrxiv.org/content/10.1101/2023.03.19.23287458v2)  
> **在线探索**：[http://txgnn.org](http://txgnn.org/)

---

## 核心问题

**传统困境**：罕见病或机制不明的疾病往往缺乏已知治疗药物，
监督学习模型面临严重的数据稀缺问题。

**TxGNN 的解法**：
```
生物医学知识图谱
  （17,080 种疾病 × 7,957 种药物 × ~25,000 个基因 + 其他生物实体）
       ↓ 图神经网络（GNN）预训练
       学习所有生物实体的低维嵌入表示
       ↓ 度量学习（Metric Learning）微调
       借助相似疾病对零样本疾病做预测
       ↓ GraphMask 可解释性
       识别哪些知识图谱路径对预测最重要
```

**预测的三种药物-疾病关系**：
- `indication`（适应症）：药物对该疾病有效  
- `contraindication`（禁忌症）：药物对该疾病有害  
- `off-label use`（超说明书）：临床使用但未正式获批

---
## Step 1：加载知识图谱数据

`TxData` 的作用：
1. 首次运行时从 Harvard Dataverse 自动下载知识图谱（约 400 MB）
2. 将原始图转换为 30 种有向关系类型
3. 按指定策略创建训练 / 验证 / 测试集划分

**`split='complex_disease'`**（论文核心评估设定）：
- 随机挑选一批疾病，将其**所有**药物关系从训练集移出，只放入测试集
- 模型在训练时对这些疾病毫无药物先验知识 → **真正的零样本场景**
- 对比：`split='random'` 让多数测试疾病在训练集中仍保留部分药物信息

> **思考**：为什么把知识图谱的边拆分为 30 种关系类型，而不是当成单一图处理？  
> 不同类型的边（如「药物抑制基因」vs「基因参与通路」）生物学含义不同，  
> 用同一变换矩阵建模会丢失这种区分信息。

In [ ]:
from txgnn import TxData, TxGNN, TxEval

# data_folder_path：数据存储目录
# 首次运行会自动下载，之后复用本地缓存
# Bug 修复：原版使用集群路径 '/n/scratch3/users/k/kh278/kg/'，已改为本地相对路径
TxData = TxData(data_folder_path = './data')

# split='complex_disease'：零样本评估划分（论文标准设定）
# seed=42：固定随机种子，保证结果可复现
# no_kg=False：使用完整知识图谱（含蛋白质、基因等辅助节点）
TxData.prepare_split(split = 'complex_disease', seed = 42, no_kg = False)

Using backend: pytorch


Found local copy...
Downloading...


  0%|          | 0.00/8.89M [00:00<?, ?iB/s]

Done!
Downloading...


  0%|          | 0.00/387M [00:00<?, ?iB/s]

Done!
First time usage... Mapping RxData raw KG to directed csv... it takes several minutes...
Iterating over relations...


  0%|          | 0/30 [00:00<?, ?it/s]

Iterating over node types...


  0%|          | 0/10 [00:00<?, ?it/s]

Creating splits... it takes several minutes...
Creating DGL graph....
Done!


**输出解读**：
- `Found local copy...`：本地已有缓存，跳过下载  
- `Iterating over relations... (0/30)`：处理 30 种关系类型（药物-疾病、基因-蛋白质等）  
- `Iterating over node types... (0/10)`：处理 10 种节点类型  
- `Creating DGL graph...`：构建 DGL 异质图（Deep Graph Library 异质图格式）

---
## Step 2：初始化模型

TxGNN 架构包含两个核心部分：

### ① GNN 编码器（基于 R-GCN）
- 对每种关系类型学习独立的变换矩阵
- 每个节点通过聚合邻居信息更新自己的嵌入向量
- 输出：每个生物实体（疾病 / 药物 / 基因）的低维嵌入

### ② 度量学习解码器（`proto=True` 时启用）
对于测试疾病（训练集中无任何已知药物）：
1. 在训练疾病中检索 `proto_num` 个最相似的疾病
2. 用这些相似疾病的嵌入**加权增强**目标疾病的嵌入
3. 用增强后的嵌入与候选药物计算匹配分数

**关键参数说明**：

| 参数 | 含义 | 推荐值 |
|------|------|-------|
| `n_hid/n_inp/n_out` | GNN 各层维度 | 100 |
| `proto=True` | 启用度量学习 | 零样本预测的核心 |
| `proto_num=3` | 检索相似疾病数量 | 3 |
| `sim_measure` | 疾病相似度计算方式 | `'all_nodes_profile'`（基于全图邻域）|
| `agg_measure='rarity'` | 聚合策略 | 罕见疾病权重更高 |
| `attention=False` | 关闭注意力层 | 训练 GraphMask 时必须关闭 |

In [ ]:
import torch

# 自动选择计算设备
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'使用设备：{device}')

TxGNN = TxGNN(data = TxData, 
              weight_bias_track = False,  # 关闭 wandb 日志（教学时关闭）
              proj_name = 'TxGNN',
              exp_name = 'TxGNN',
              device = device
              )

# 如果已有保存的模型权重，可以直接加载跳过训练：
# TxGNN.load_pretrained('./model_ckpt')

TxGNN.model_initialize(
    n_hid = 100,                        # 隐层维度
    n_inp = 100,                        # 输入维度
    n_out = 100,                        # 输出维度
    proto = True,                       # 启用度量学习（零样本预测核心）
    proto_num = 3,                      # 检索相似疾病数量
    attention = False,                  # 关闭注意力（GraphMask 要求）
    sim_measure = 'all_nodes_profile',  # 基于全图邻域计算疾病相似度
    bert_measure = 'disease_name',      # BERT 文本相似度（备用）
    agg_measure = 'rarity',             # 按疾病稀有程度加权聚合
    num_walks = 200,                    # 随机游走次数
    walk_mode = 'bit',
    path_length = 2                     # 随机游走路径长度
)

> **消融实验**：将 `proto=True` 改为 `proto=False`，得到无度量学习的基线（普通 GNN）。  
> 论文结论：在 `complex_disease` 划分上，`proto=True` 的 Macro AUROC 显著更高，  
> 说明度量学习对零样本泛化至关重要。

---
## Step 3：预训练（Pre-training）

**预训练目标**：在知识图谱的**所有 30 种关系类型**上做链接预测。

类比 NLP：就像用大规模语料预训练 BERT，让模型先学到通用的实体表示，再针对具体任务微调。

- **正样本**：图中真实存在的边  
- **负样本**：随机采样的不存在的边  
- **目标**：正样本得分 > 负样本得分（负采样 + 二元交叉熵）

> **课堂演示说明**：完整预训练需要数小时，以下代码已注释。  
> 若有预训练权重，可在 Step 2 中取消 `load_pretrained` 的注释直接加载。  
> 权重下载：https://drive.google.com/file/d/1fxTFkjo2jvmz9k6vesDbCeucQjGRojLj/view

In [ ]:
## 预训练（演示时跳过，输出较长）
## 如需运行，取消下方注释；正式实验建议 n_epoch=10~20

# TxGNN.pretrain(n_epoch = 2, 
#                learning_rate = 1e-3,
#                batch_size = 1024, 
#                train_print_per_n = 20)

print('预训练已跳过，直接进入微调')
print('提示：没有预训练权重时，微调仍可运行，但性能低于论文结果')

---
## Step 4：微调（Fine-tuning）

微调阶段专注于**药物-疾病关系**，训练度量学习头。

**评估指标解读**：

| 指标 | 含义 | 重要性 |
|------|------|-------|
| `Micro AUROC` | 所有（药物,疾病）对等权重 | 整体性能 |
| `Macro AUROC` | 各疾病等权重平均 | **论文主要指标**，更关注罕见病 |
| `AUPRC` | 精确率-召回率曲线下面积 | 正负样本不平衡时更敏感 |

**预期变化**：AUROC 从约 0.50（随机水平）随 epoch 逐步上升至 0.60~0.75+

In [ ]:
## 演示时 n_epoch=30，正式实验改为 n_epoch=500
## CPU 模式下可改为 n_epoch=5（约2分钟）
TxGNN.finetune(n_epoch = 30, 
               learning_rate = 5e-4,
               train_print_per_n = 5,   # 每5轮打印训练指标
               valid_per_n = 20)        # 每20轮评估验证集

Epoch: 0 LR: 0.00050 Loss 0.6938, Train Micro AUROC 0.5109 Train Micro AUPRC 0.5127 Train Macro AUROC 0.5019 Train Macro AUPRC 0.5073
----- AUROC Performance in Each Relation -----
('drug', 'contraindication', 'disease'): 0.5349347764149285
('drug', 'indication', 'disease'): 0.5475017777377161
('drug', 'off-label use', 'disease'): 0.4750877230935641
('disease', 'rev_contraindication', 'drug'): 0.5200441594466088
('disease', 'rev_indication', 'drug'): 0.42414272145771154
('disease', 'rev_off-label use', 'drug'): 0.5098947539210383
----- AUPRC Performance in Each Relation -----
('drug', 'contraindication', 'disease'): 0.5439511651288017
('drug', 'indication', 'disease'): 0.551160435487897
('drug', 'off-label use', 'disease'): 0.4725924166335391
('disease', 'rev_contraindication', 'drug'): 0.5207788369454536
('disease', 'rev_indication', 'drug'): 0.4539515055664601
('disease', 'rev_off-label use', 'drug'): 0.5012481020151309
----------------------------------------------
Validation.....
E

**观察要点**：
1. 训练开始时 AUROC ≈ 0.50（随机水平），随 epoch 稳步上升  
2. `indication` 的 AUROC 通常高于 `off-label use`——适应症标签更可靠  
3. 验证集行末 `(Best Macro AUROC X.XX)` 记录当前最佳值，用于早停  
4. 若验证集 AUROC 不再上升而训练集持续上升，说明开始过拟合

> **思考**：为什么同时预测 `indication` 和 `contraindication`，而非只预测适应症？  
> 禁忌症同样反映药物与疾病的相关性，能帮助 GNN 学到更准确的实体表示。

---
## Step 5：保存与加载模型

训练完成后保存权重，下次可直接加载跳过训练。

In [ ]:
import os

# 保存模型到 ./model_ckpt 目录
os.makedirs('./model_ckpt', exist_ok=True)  # 确保目录存在
TxGNN.save_model('./model_ckpt')
print('模型已保存')

# 验证加载（下次可直接从此处开始，跳过训练）
TxGNN.load_pretrained('./model_ckpt')
print('模型加载验证成功')

---
## Step 6：可解释性分析（GraphMask）

### 为什么需要可解释性？
医疗决策必须有理可循。GraphMask 回答：
> *「模型预测『药物 A 可治疗疾病 B』时，依赖的是知识图谱中哪些路径？」*

### GraphMask 原理
```
对 GNN 每一层的每条边，学习一个 gate 值（0~1）
    gate ≈ 1：该边对预测重要，保留
    gate ≈ 0：该边对预测不重要，屏蔽

双重优化目标：
    ① 保持预测性能（AUROC 下降 ≤ allowance=0.5%）
    ② 最小化 gate=1 的边数（稀疏性约束）
```
训练后，gate 值高的边 = **对药物重定向预测最重要的知识图谱路径**

例：若「药物 → 靶点基因 → 疾病相关基因 → 疾病」路径的 gate 值高，  
说明**靶点机制**是该预测的主要依据。

> **Bug 修复**：原版代码中 `epochs_per_layer` 参数重复出现（`=3` 和 `=5`），  
> Python 实际取最后一个值 `=5`，但存在歧义，已清理。

In [ ]:
# Bug 修复：原版中 epochs_per_layer 参数重复出现（=3 和 =5），
# Python 取最后一个值 =5，但代码存在歧义，已清理
TxGNN.train_graphmask(
    relation = 'indication',
    learning_rate = 3e-4,
    allowance = 0.005,       # 允许的最大性能损失（0.5%）
    epochs_per_layer = 5,    # 每个 GNN 层训练 5 轮
    penalty_scaling = 1,     # 稀疏性惩罚系数
    valid_per_n = 20
)

# 提取并保存各边的 gate 值
# 输出：./model_ckpt/graphmask_output_indication.pkl
output = TxGNN.retrieve_save_gates('./model_ckpt')

# 保存 GraphMask 模型权重
TxGNN.save_graphmask_model('./graphmask_model_ckpt')

**加载已训练的 GraphMask**（下次运行时可跳过重新训练）：
```python
TxGNN.load_pretrained_graphmask('./graphmask_model_ckpt')
```

> **思考**：GraphMask 的稀疏性约束与 L1 正则化有什么相似之处？  
> 两者都鼓励权重趋向零，但 GraphMask 作用于**边的重要性**，而非模型参数本身。

---
## Step 7：模型评估

`TxEval` 采用「**疾病中心**」评估框架：
- 对每个测试疾病，将**全部候选药物**按预测分数排序
- 评估已知药物（适应症）是否排在前列
- 聚合所有测试疾病的 AUROC / AUPRC

这比简单二分类更贴近临床实际——医生需要的是排名靠前的候选药物清单。

In [ ]:
from txgnn import TxEval

# 初始化评估器，传入训练好的 TxGNN 模型
TxEval = TxEval(model = TxGNN)

In [ ]:
# 评估特定疾病（disease_idxs 为知识图谱中的节点编号）
result = TxEval.eval_disease_centric(
    disease_idxs = [12661.0, 11318.0], 
    relation = 'indication', 
    save_result = False
)

# 评估整个测试集（约 66 种零样本疾病）
# Bug 修复：原版误写为 RxEval，已修正为 TxEval
result = TxEval.eval_disease_centric(
    disease_idxs = 'test_set', 
    show_plot = False, 
    verbose = True, 
    save_result = True,
    return_raw = False
)

**结果解读**：
- `Macro AUROC ≈ 0.75+`（充分训练后）：模型对多数零样本疾病能给出合理排名  
- 不同疾病的 AUROC 差异较大：图中邻居（基因/通路）越丰富的疾病预测越准  
- `off-label use` 的 AUROC 通常低于 `indication`：超说明书标签噪声更大

> **Bug 修复**：原版代码中将 `TxEval` 误写为 `RxEval`，已修正。

---
## Step 8：查看测试集疾病列表

以下列出测试集中所有零样本疾病的节点编号（知识图谱 ID）。  
这些疾病在训练时没有任何已知药物——模型必须完全依赖知识图谱结构和相似疾病来预测。

In [ ]:
# 获取测试集中所有疾病的节点编号
# 这些疾病在训练时没有任何已知药物（零样本场景）
test_idxs = TxEval.retrieve_disease_idxs_test_set('indication')
print(f'测试集共 {len(test_idxs)} 种零样本疾病')
print(test_idxs)

array([ 7701., 12661., 11195., 16569.,  6343.,  1479., 12621., 14416.,
       13557., 16501., 13446., 13610.,  7105.,  9716.,  5822., 12320.,
       10023., 14513., 12929.,  5683.,  9950., 14661.,  8395.,  7990.,
        6031.,  6255., 14409.,  9589., 11885.,  4285.,  6377., 13130.,
        7773.,  9741., 13381., 15355.,  4536., 12891.,  9087., 14873.,
        6185.,  1812., 15301.,  6474.,  3381.,  6599., 16965.,  7880.,
       16698.,  6252.,  5764.,  4192.,  4467., 13107., 14501., 11318.,
        3452., 11349., 15911., 15100.,  1417.,  4152.,  1319.,  2220.,
       15989.,  8973.])

---
## 总结与延伸思考

### 完整流程回顾
```
TxData（数据加载 + complex_disease 划分）
    → TxGNN.model_initialize（R-GCN + 度量学习头）
    → TxGNN.pretrain（全图 30 种关系链接预测，可跳过）
    → TxGNN.finetune（药物-疾病关系，zero-shot 泛化）
    → TxGNN.train_graphmask（GraphMask 稀疏门控，可解释性）
    → TxEval.eval_disease_centric（疾病中心 AUROC/AUPRC 评估）
```

### 本 Notebook 修复的三处原版 Bug
| # | 位置 | 原始代码 | 修复后 |
|---|------|---------|-------|
| 1 | Step 1 | `data_folder_path='/n/scratch3/...'`（集群路径） | `'./data'` |
| 2 | Step 6 | `epochs_per_layer=3` 和 `=5` 同时出现 | 保留 `=5`，删除 `=3` |
| 3 | Step 7 | `RxEval.eval_disease_centric(...)` | `TxEval.eval_disease_centric(...)` |

### 延伸思考
1. **知识图谱质量**：若某疾病在 KG 中完全孤立（无任何邻居），度量学习还能预测吗？  
2. **少样本扩展**：如果测试疾病有 1~2 个已知药物，如何无需重训练地利用这些信息？  
3. **可解释性验证**：GraphMask 找到的重要路径如何与临床医学知识对照验证？  
4. **领域实验**：尝试 `split='cardiovascular'` vs `split='diabetes'`，  
   哪个领域的零样本预测更难？为什么？

### 参考资料
- 论文：[TxGNN MedRxiv](https://www.medrxiv.org/content/10.1101/2023.03.19.23287458v2)  
- 代码：[github.com/mims-harvard/TxGNN](https://github.com/mims-harvard/TxGNN)  
- 在线探索器：[txgnn.org](http://txgnn.org/)